In [ ]:
# ==========================================
# STAGE 1: FULL COLAB SETUP & OCR PIPELINE (WITH TEXT EXPORT & EVALUATION)
# ==========================================

# ------------------------------------------
# 0. DOWNLOAD PROJECT FILES AND DATA (GITHUB)
# ------------------------------------------
import os

REPO_URL = "https://github.com/yemenoglu/Handwritten-Circuit-Recognition-to-Netlist.git"
REPO_DIR = "Handwritten-Circuit-Recognition-to-Netlist"

if not os.path.exists(REPO_DIR):
    print("📥 Fetching project files and sample data from GitHub...")
    !git clone {REPO_URL}
else:
    print("✅ Project files already exist.")

colab_repo_path = f"/content/{REPO_DIR}"
if os.path.exists(colab_repo_path):
    os.chdir(colab_repo_path)
    print(f"📂 Changed directory to: {os.getcwd()}")

# Klasörleri doğru dizine girdikten sonra oluşturuyoruz
os.makedirs("output/visualizations", exist_ok=True)
os.makedirs("output/ocr_data", exist_ok=True)
os.makedirs("output/no_text_images", exist_ok=True)
print("📁 Output folders ready.")

# ------------------------------------------
# 1. DEPENDENCIES & WEIGHTS
# ------------------------------------------
print("\n📦 Installing dependencies...")
!pip install -q svg_schematic cairosvg ultralytics gdown

import tensorflow as tf

def download_weights():
    models = {
        "yolo_detector.pt": "1wzpIBN6jv9NRJRu7LYF4A86RunyeBBFX",
        "component_classifier.keras": "1Gp1Bm11PeT69aRV-b7ekxn2gmwVFYuMy",
        "crnn_text_model.pth": "11IOPbIkYQYIbQw7col8pSmoZPggWdDal"
    }
    os.makedirs("weights", exist_ok=True)
    print("\n📥 Checking and downloading model weights...")
    for name, file_id in models.items():
        if not os.path.exists(f"weights/{name}"):
            print(f"Downloading {name}...")
            !gdown {file_id} -O weights/{name}
        else:
            print(f"✅ {name} already exists.")

download_weights()

# ------------------------------------------
# 2. IMPORTS & CONFIGURATION
# ------------------------------------------
import cv2
import math
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms
from PIL import Image
import json
import re
import networkx as nx
from ultralytics import YOLO
from skimage import morphology
import matplotlib.pyplot as plt

TEST_FOLDER = "data/samples"
GROUND_TRUTH_FILE = "data/ground_truth_demo.txt"
OUTPUT_NO_TEXT_FOLDER = "output/no_text_images"
VISUALIZATION_FOLDER = "output/visualizations"

YOLO_MODEL_PATH = "weights/yolo_detector.pt"
CRNN_MODEL_PATH = "weights/crnn_text_model.pth"

ERASING_SHRINK_RATIO = 0.1

alphabet = ['-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'C', 'D', 'F', 'H', 'L', 'M', 'O', 'R', 'V', 'h', 'k', 'm', 'n', 'p', 'u']
ALPHABET_SIZE = len(alphabet)
idx_to_char = {idx: char for idx, char in enumerate(alphabet)}

# ------------------------------------------
# 3. HELPER & METRIC FUNCTIONS
# ------------------------------------------
def standardize_text(text):
    if not text: return ""
    text = str(text).strip()
    text = re.sub(r'(?i)ohm', 'Ohm', text)
    text = re.sub(r'v$', 'V', text)
    text = re.sub(r'f$', 'F', text)
    text = re.sub(r'h$', 'H', text)
    return text

def levenshtein_distance(s1, s2):
    if len(s1) < len(s2): return levenshtein_distance(s2, s1)
    if len(s2) == 0: return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def pair_predictions_to_gt(preds, gts):
    paired_errors = []
    used_gts = set()
    for pred in preds:
        best_gt = None
        best_dist = float('inf')
        best_gt_idx = -1
        for i, gt in enumerate(gts):
            if i in used_gts: continue
            dist = levenshtein_distance(pred, gt)
            if dist < best_dist:
                best_dist = dist
                best_gt = gt
                best_gt_idx = i
        if best_gt is not None:
            used_gts.add(best_gt_idx)
            paired_errors.append((pred, best_gt, best_dist))
        else:
            paired_errors.append((pred, "", len(pred)))
    for i, gt in enumerate(gts):
        if i not in used_gts:
            paired_errors.append(("", gt, len(gt)))
    return paired_errors

def find_center(x1, y1, x2, y2): return ((x1 + x2) / 2, (y1 + y2) / 2)
def calculate_distance(m1, m2): return math.sqrt((m1[0] - m2[0])**2 + (m1[1] - m2[1])**2)

class WhitePaddedCenterResize(object):
    def __init__(self, target_size=(128, 32)): self.target_size = target_size
    def __call__(self, img):
        w, h = img.size
        if h == 0: return Image.new('L', self.target_size, 255)
        ratio = h / self.target_size[1]
        new_w = int(w / ratio)
        if new_w > self.target_size[0]: new_w = self.target_size[0]
        resized_img = img.resize((new_w, self.target_size[1]), Image.Resampling.LANCZOS)
        new_img = Image.new('L', self.target_size, 255)
        x_offset = (self.target_size[0] - new_w) // 2
        new_img.paste(resized_img, (x_offset, 0))
        return new_img

transform = transforms.Compose([
    WhitePaddedCenterResize((128, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class CircuitReaderCRNN(nn.Module):
    def __init__(self, num_chars):
        super(CircuitReaderCRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        )
        self.rnn = nn.LSTM(input_size=1024, hidden_size=256, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256 * 2, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        b, c, h, w = conv_out.size()
        conv_out = conv_out.view(b, c * h, w).permute(0, 2, 1)
        rnn_out, _ = self.rnn(conv_out)
        return self.fc(rnn_out).permute(1, 0, 2)

def ctc_decoder(prediction_tensor, idx_to_char):
    _, max_indices = torch.max(prediction_tensor, dim=1)
    decoded_text = []
    previous_index = None
    for index in max_indices:
        index = index.item()
        if index != 0 and index != previous_index: decoded_text.append(idx_to_char[index])
        previous_index = index
    return "".join(decoded_text)

# ------------------------------------------
# 4. INITIALIZE MODELS
# ------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🚀 Initializing models... (Device: {device.type.upper()})")

yolo_model = YOLO(YOLO_MODEL_PATH)
crnn_model = CircuitReaderCRNN(num_chars=ALPHABET_SIZE).to(device)
crnn_model.load_state_dict(torch.load(CRNN_MODEL_PATH, map_location=device))
crnn_model.eval()

gt_data = {}
try:
    with open(GROUND_TRUTH_FILE, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        if not content.startswith('{'): content = '{' + content + '}'
        gt_data = eval(content, {"nx": nx})
    print("✅ Ground Truth file loaded successfully.")
except Exception as e:
    print(f"⚠️ WARNING: Could not read Ground Truth file. Error: {e}")

# ------------------------------------------
# 5. PROCESS IMAGES (BATCH WITH VISUALIZATION & EXPORT)
# ------------------------------------------
print("\n" + "="*60)
print(f"📂 STARTING BATCH OCR & PREPROCESSING")
print("="*60)

total_chars = 0
total_char_errors = 0
total_words = 0
total_word_errors = 0
exact_matches = 0

if not os.path.exists(TEST_FOLDER):
    print(f"❌ TEST_FOLDER '{TEST_FOLDER}' not found.")
    image_files = []
else:
    image_files = [f for f in os.listdir(TEST_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

ocr_master_dict = {}

for img_idx, filename in enumerate(image_files):
    vis_steps = {}
    vis_crops = []
    img_ocr_data = []

    img_path = os.path.join(TEST_FOLDER, filename)
    original_img = cv2.imread(img_path)
    if original_img is None: continue

    vis_steps['input'] = original_img.copy()

    print(f"\n[{img_idx+1}/{len(image_files)}] Processing: {filename}")
    ori_h, ori_w = original_img.shape[:2]

    scale_x_down = 1000.0 / ori_w
    scale_y_down = 750.0 / ori_h
    scale_x_up = ori_w / 1000.0
    scale_y_up = ori_h / 750.0

    img_resized = cv2.resize(original_img, (1000, 750), interpolation=cv2.INTER_CUBIC)

    text_results = yolo_model.predict(source=img_path, conf=0.35, verbose=False)
    text_boxes = text_results[0].boxes

    print(f"   [DEBUG] YOLO found {len(text_boxes)} text box(es).")

    detection_img = img_resized.copy()
    img_no_text = img_resized.copy()
    original_no_text = original_img.copy()

    for t_box in text_boxes:
        tx1, ty1, tx2, ty2 = map(int, t_box.xyxy[0])

        cv2.rectangle(detection_img, (int(tx1 * scale_x_down), int(ty1 * scale_y_down)),
                                     (int(tx2 * scale_x_down), int(ty2 * scale_y_down)), (0, 0, 255), 3)

        box_width = tx2 - tx1
        shrink_px = int(box_width * ERASING_SHRINK_RATIO)
        erase_tx1 = tx1 + shrink_px
        erase_tx2 = tx2 - shrink_px

        if erase_tx1 >= erase_tx2:
            erase_tx1 = tx1
            erase_tx2 = tx2

        cv2.rectangle(img_no_text, (int(erase_tx1 * scale_x_down), int(ty1 * scale_y_down)),
                                   (int(erase_tx2 * scale_x_down), int(ty2 * scale_y_down)), (255, 255, 255), -1)
        cv2.rectangle(original_no_text, (erase_tx1, ty1), (erase_tx2, ty2), (255, 255, 255), -1)

    vis_steps['detections'] = detection_img
    save_path = os.path.join(OUTPUT_NO_TEXT_FOLDER, filename)
    cv2.imwrite(save_path, original_no_text)
    vis_steps['final_masked'] = original_no_text.copy()

    # Image processing for heuristic rules
    img_gray = cv2.cvtColor(img_no_text, cv2.COLOR_BGR2GRAY)
    img_blur = cv2.GaussianBlur(img_gray, (5,5), 0)
    imgTres = cv2.adaptiveThreshold(img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 49, 29)
    imgSkel = morphology.skeletonize(cv2.bitwise_not(imgTres) / 255)
    imgSkel = (imgSkel.astype(np.float32) * 255).astype(np.uint8)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (30, 25))
    img_close = cv2.morphologyEx(imgSkel, cv2.MORPH_CLOSE, kernel_close, iterations=2)
    img_blob = cv2.erode(img_close, np.ones((3, 3), np.uint8), iterations=2)

    contours, _ = cv2.findContours(img_blob, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    comp_boxes_custom = []

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if max(w, h) < 45: continue
        square = (y, y+h, x-h//2+w//2, x+h//2+w//2) if h > w else (y-w//2+h//2, y+w//2+h//2, x, x+w)
        y1, y2, x1, x2 = square
        y1, y2 = max(0, y1), min(img_resized.shape[0], y2)
        x1, x2 = max(0, x1), min(img_resized.shape[1], x2)
        comp_boxes_custom.append({
            'xyxy': [x1, y1, x2, y2],
            'orientation': 'VERTICAL' if h > w else 'HORIZONTAL'
        })

    predicted_texts = []

    for i, t_box in enumerate(text_boxes):
        tx1, ty1, tx2, ty2 = map(int, t_box.xyxy[0])
        t_center = find_center(tx1, ty1, tx2, ty2)

        # Calculate center relative to the 1000x750 scale for Stage 2 matching
        cx_res = ((tx1 + tx2) / 2.0) * scale_x_down
        cy_res = ((ty1 + ty2) / 2.0) * scale_y_down

        closest_dist = float('inf')
        closest_orientation = "UNKNOWN"

        for c_box in comp_boxes_custom:
            cx1, cy1, cx2, cy2 = c_box['xyxy']
            ori_cx1, ori_cy1 = cx1 * scale_x_up, cy1 * scale_y_up
            ori_cx2, ori_cy2 = cx2 * scale_x_up, cy2 * scale_y_up
            c_center = find_center(ori_cx1, ori_cy1, ori_cx2, ori_cy2)
            dist = calculate_distance(t_center, c_center)
            if dist < closest_dist:
                closest_dist = dist
                closest_orientation = c_box['orientation']

        box_width = tx2 - tx1
        max_shave = int(25 * scale_x_up)

        if closest_orientation == "VERTICAL":
            crop_left = min(max_shave, int(box_width * 0.35))
            crop_right = min(max_shave, int(box_width * 0.35))
            new_tx1 = min(tx1 + crop_left, tx2 - 5)
            new_tx2 = max(tx2 - crop_right, new_tx1 + 5)
            new_ty1, new_ty2 = ty1, ty2
        else:
            new_tx1, new_tx2, new_ty1, new_ty2 = tx1, tx2, ty1, ty2

        ori_tx1 = max(0, int(new_tx1))
        ori_ty1 = max(0, int(new_ty1))
        ori_tx2 = min(ori_w, int(new_tx2))
        ori_ty2 = min(ori_h, int(new_ty2))

        cropped_cv2 = original_img[ori_ty1:ori_ty2, ori_tx1:ori_tx2]

        if cropped_cv2.size == 0: continue

        cropped_pil = Image.fromarray(cv2.cvtColor(cropped_cv2, cv2.COLOR_BGR2RGB)).convert('L')
        tensor_img = transform(cropped_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            prediction_output = crnn_model(tensor_img).squeeze(1)

        decoded_text = ctc_decoder(prediction_output, idx_to_char)

        if decoded_text:
            predicted_texts.append(decoded_text)
            img_ocr_data.append({"text": decoded_text, "center": [cx_res, cy_res]})

        if len(vis_crops) < 5:
            vis_crops.append({'img': cropped_cv2.copy(), 'text': decoded_text, 'orient': closest_orientation})

    ocr_master_dict[filename] = img_ocr_data

    # ------------------------------------------
    # 6. GT EVALUATION
    # ------------------------------------------
    gt_texts = []
    if filename in gt_data:
        G = gt_data[filename].get("graph", None)
        if G is not None and isinstance(G, nx.Graph):
            for u, v, data in G.edges(data=True):
                lbl = str(data.get("label", "")).strip()
                if lbl: gt_texts.append(lbl)

    predicted_texts = [standardize_text(p) for p in predicted_texts]
    gt_texts = [standardize_text(gt) for gt in gt_texts]

    if gt_texts or predicted_texts:
        print(f"   Predictions : {predicted_texts}")
        print(f"   GroundTruth : {gt_texts}")

        paired_results = pair_predictions_to_gt(predicted_texts, gt_texts)
        img_char_errors = 0
        img_chars_total = sum(len(gt) for gt in gt_texts)
        if img_chars_total == 0: img_chars_total = 1

        for pred, gt, dist in paired_results:
            img_char_errors += dist
            total_words += 1
            if dist == 0:
                exact_matches += 1
                print(f"   ✅ MATCH   : '{pred}'")
            else:
                total_word_errors += 1
                print(f"   ❌ ERROR   : Found='{pred}' | Expected='{gt}' | Dist={dist} chars")

        total_chars += img_chars_total
        total_char_errors += img_char_errors

    # Visualization
    def bgr2rgb(img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if len(img.shape) == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    fig = plt.figure(figsize=(18, 10))
    plt.suptitle(f"Pipeline & OCR Predictions: {filename}", fontsize=16, fontweight='bold')

    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(bgr2rgb(vis_steps['input']))
    ax1.set_title("1. Original Input Image")
    ax1.axis('off')

    ax2 = plt.subplot(2, 3, 2)
    ax2.imshow(bgr2rgb(vis_steps['detections']))
    ax2.set_title(f"2. YOLO Detections ({len(text_boxes)} boxes)")
    ax2.axis('off')

    ax3 = plt.subplot(2, 3, 3)
    ax3.imshow(bgr2rgb(vis_steps['final_masked']))
    ax3.set_title("3. Final Preprocessed Image")
    ax3.axis('off')

    if vis_crops:
        num_crops = len(vis_crops)
        for i, crop_info in enumerate(vis_crops):
            ax_crop = plt.subplot(2, num_crops, num_crops + i + 1)
            ax_crop.imshow(bgr2rgb(crop_info['img']))
            title_str = f"Pred: '{crop_info['text']}'\n(Rule: {crop_info['orient'][:4]})"
            ax_crop.set_title(title_str, fontsize=12, fontweight='bold', color='darkred')
            ax_crop.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    vis_save_path = os.path.join(VISUALIZATION_FOLDER, f"{os.path.splitext(filename)[0]}_ocr_showcase.png")
    plt.savefig(vis_save_path, dpi=150)
    plt.show()
    plt.close(fig)

# Save OCR dictionary for Stage 2
ocr_json_path = "output/ocr_data/ocr_results.json"
with open(ocr_json_path, "w", encoding='utf-8') as f:
    json.dump(ocr_master_dict, f, indent=4)
print(f"\n✅ All OCR data securely exported to: {ocr_json_path} for Component Matching.")

# ------------------------------------------
# 7. PERFORMANCE REPORT
# ------------------------------------------
print("\n" + "="*60)
print("🏆 OVERALL OCR PERFORMANCE REPORT")
print("="*60)

if total_chars > 0 and total_words > 0:
    cer = (total_char_errors / total_chars) * 100
    wer = (total_word_errors / total_words) * 100
    word_accuracy = (exact_matches / total_words) * 100

    print(f"Total Images Processed      : {len(image_files)}")
    print(f"Total Labels Matched        : {total_words}")
    print(f"Total Ground Truth Chars    : {total_chars}\n")

    print(f"✅ Exact Label Matches      : {exact_matches}")
    print(f"❌ Erroneous Labels         : {total_word_errors}")
    print(f"❌ Total Character Errors   : {total_char_errors}\n")

    print(f"📈 Word Accuracy            : % {word_accuracy:.2f}  (Exact match)")
    print(f"📉 WER (Word Error Rate)    : % {wer:.2f}")
    print(f"📉 CER (Character Error Rate): % {cer:.2f}")

    # ------------------------------------------
# 7. PERFORMANCE REPORT & ERROR ANALYSIS
# ------------------------------------------
import difflib
from collections import Counter
import pandas as pd
from IPython.display import display

print("\n" + "="*60)
print("🏆 OVERALL OCR PERFORMANCE REPORT & ERROR ANALYSIS")
print("="*60)

# Global karakter hatası takibi
char_misses = Counter()
char_false_alarms = Counter()

if total_chars > 0 and total_words > 0:
    cer = (total_char_errors / total_chars) * 100
    wer = (total_word_errors / total_words) * 100
    word_accuracy = (exact_matches / total_words) * 100

    print(f"Total Images Processed      : {len(image_files)}")
    print(f"Total Labels Matched        : {total_words}")
    print(f"Total Ground Truth Chars    : {total_chars}\n")
    print(f"✅ Exact Label Matches      : {exact_matches}")
    print(f"❌ Erroneous Labels         : {total_word_errors}")
    print(f"📈 Word Accuracy            : % {word_accuracy:.2f}  (Exact match)")
    print(f"📉 CER (Character Error Rate): % {cer:.2f}\n")

    # Tüm tahminleri ve gerçekleri tekrar dönüp karakter analizi yapıyoruz
    for img_data in image_files:
        filename = img_data
        gt_texts_img = []
        if filename in gt_data:
            G = gt_data[filename].get("graph", None)
            if G is not None:
                for u, v, data in G.edges(data=True):
                    lbl = str(data.get("label", "")).strip()
                    if lbl: gt_texts_img.append(standardize_text(lbl))

        pred_texts_img = [standardize_text(p["text"]) for p in ocr_master_dict.get(filename, [])]
        paired = pair_predictions_to_gt(pred_texts_img, gt_texts_img)

        for pred, gt, dist in paired:
            if dist > 0:
                # Karakter karakter farkı bul
                diff = difflib.ndiff(gt, pred)
                for d in diff:
                    if d.startswith('- '): # GT'de var, tahminde yok/yanlış (Kaçırılan)
                        char_misses[d[2]] += 1
                    elif d.startswith('+ '): # Tahminde var, GT'de yok (Yanlış uydurulan)
                        char_false_alarms[d[2]] += 1

    print("▶ OCR CHARACTER-LEVEL ERROR ANALYSIS (En Çok Karıştırılan Karakterler):")
    error_data = []
    for char, count in char_misses.most_common(10):
        false_count = char_false_alarms.get(char, 0)
        error_data.append({"Character": char, "Missed/Misclassified": count, "Falsely Inserted": false_count})

    display(pd.DataFrame(error_data))

In [ ]:
# ==========================================
# STAGE 2: COMPONENT DETECTION, VALUE MATCHING, CAD GENERATION & EVALUATION
# ==========================================
import cv2
import math
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import tensorflow as tf
from skimage import morphology
from scipy.spatial.distance import euclidean
from collections import Counter
from IPython.display import display

# 1. PIPELINE CHECK
OCR_OUTPUT_FOLDER = "output/no_text_images"
OCR_RESULTS_FILE = "output/ocr_data/ocr_results.json"

if not os.path.exists(OCR_OUTPUT_FOLDER) or not os.path.exists(OCR_RESULTS_FILE):
    raise FileNotFoundError("❌ ERROR: Please run Stage 1 (OCR script) first!")

print("✅ Preprocessed images and OCR data found. Proceeding with pipeline...")

os.makedirs("output/netlists", exist_ok=True)
os.makedirs("output/cad_drawings", exist_ok=True)

# 2. PARAMETERS & MODELS
TEST_FOLDER = OCR_OUTPUT_FOLDER
GROUND_TRUTH_FILE = "data/ground_truth_demo.txt"
KERAS_CLASSIFIER_PATH = "weights/component_classifier.keras"

IMG_HEIGHT, IMG_WIDTH = 160, 160
CLASS_NAMES = [
    '0_diode_H_white', '0_diode_V_white', '1_resistor_H_white', '1_resistor_V_white',
    '2_inductor_H_white', '2_inductor_V_white', '3_capacitor_H_white', '3_capacitor_V_white', '5_DC_power_white'
]

COMP_MAP = {
    '0_diode_H_white': 'diode', '0_diode_V_white': 'diode',
    '1_resistor_H_white': 'resistor', '1_resistor_V_white': 'resistor',
    '2_inductor_H_white': 'inductor', '2_inductor_V_white': 'inductor',
    '3_capacitor_H_white': 'capacitor', '3_capacitor_V_white': 'capacitor',
    '5_DC_power_white': 'source'
}

print("\n🧠 Loading Keras component classifier...")
model = tf.keras.models.load_model(KERAS_CLASSIFIER_PATH)

# Load OCR Data
with open(OCR_RESULTS_FILE, 'r', encoding='utf-8') as f:
    ocr_master_dict = json.load(f)

# Load Ground Truth Data
ground_truth_db = {}
try:
    with open(GROUND_TRUTH_FILE, 'r', encoding='utf-8') as f:
        content = f.read().strip()
        if not content.startswith('{'): content = '{' + content + '}'
        ground_truth_db = eval(content, {"nx": nx})
    print("✅ Ground Truth file loaded successfully for Stage 2.")
except Exception as e:
    print(f"⚠️ WARNING: Could not read Ground Truth file in Stage 2. Error: {e}")

# 3. HELPER FUNCTIONS
def preprocess_component_crop(crop):
    if len(crop.shape) == 3: crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    crop = cv2.resize(crop, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_AREA)
    crop = crop.astype("float32")
    return np.expand_dims(np.expand_dims(crop, axis=-1), axis=0)

def snap_to_grid(y, x, grid_size=15):
    return (int(round(y / grid_size) * grid_size), int(round(x / grid_size) * grid_size))

def get_terminal_locations(comp_info):
    y1, y2, x1, x2 = map(int, comp_info['location'])
    if comp_info['orientation'] == 'h': return [(y1 + (y2 - y1) // 2, x1), (y1 + (y2 - y1) // 2, x2)]
    else: return [(y1, x1 + (x2 - x1) // 2), (y2, x1 + (x2 - x1) // 2)]

def scale_gt_coords(gt_coords_xy, orig_w, orig_h, target_w=1000, target_h=750):
    scale_x, scale_y = target_w / orig_w, target_h / orig_h
    return [(int(y * scale_y), int(x * scale_x)) for x, y in gt_coords_xy]

def calc_node_accuracy(pred_nodes, gt_coords_orig, orig_w, orig_h, threshold=20):
    gt_scaled = scale_gt_coords(gt_coords_orig, orig_w, orig_h)
    correct, matched_gt = 0, set()
    for p in pred_nodes:
        py, px = p['location']
        min_d, best_gt = float('inf'), -1
        for i, (gy, gx) in enumerate(gt_scaled):
            if i in matched_gt: continue
            d = euclidean((py, px), (gy, gx))
            if d < min_d: min_d, best_gt = d, i
        if min_d <= threshold:
            correct += 1
            matched_gt.add(best_gt)
    return len(gt_scaled), len(pred_nodes), correct, (correct / len(gt_scaled) * 100) if len(gt_scaled) > 0 else 0

def calc_component_accuracy(pred_components, G_gt):
    gt_types = [data.get('type') for u, v, data in G_gt.edges(data=True)]
    pred_types = [cdata.get('type') for cdata in pred_components.values()]
    gt_counter, pred_counter = Counter(gt_types), Counter(pred_types)
    correct_count = sum(min(gt_count, pred_counter.get(ctype, 0)) for ctype, gt_count in gt_counter.items())
    total_gt = len(gt_types)
    return (correct_count / total_gt * 100) if total_gt > 0 else 0

def eval_isomorphism(G_pred, G_gt):
    struct_iso = nx.is_isomorphic(G_pred, G_gt)
    exact_iso = nx.is_isomorphic(G_pred, G_gt, edge_match=lambda e1, e2: e1.get('type') == e2.get('type'))
    return struct_iso, exact_iso

def generate_json_netlist(G_ext, pred_nodes):
    node_locations = {n['name']: n['location'] for n in pred_nodes}
    node_mapping = {n: i for i, n in enumerate(G_ext.nodes())}
    netlist_dict = {}

    for node in G_ext.nodes():
        node_id = node_mapping[node]
        netlist_dict[node_id] = []

        for neighbor in G_ext.neighbors(node):
            edge_data = G_ext.get_edge_data(node, neighbor)
            comp_type = edge_data.get('mapped_type', 'unknown')
            if comp_type == 'unknown': continue

            y1, y2, x1, x2 = edge_data['location']
            center_x, center_y = int((x1 + x2) / 2), int((y1 + y2) / 2)
            neighbor_id = node_mapping[neighbor]

            node_loc = node_locations[node]
            neigh_loc = node_locations[neighbor]

            if edge_data['orientation'] == 'h':
                ordered_nodes = [node_id, neighbor_id] if node_loc[1] < neigh_loc[1] else [neighbor_id, node_id]
            else:
                ordered_nodes = [node_id, neighbor_id] if node_loc[0] < neigh_loc[0] else [neighbor_id, node_id]

            component_entry = {
                "component": {
                    "type": comp_type,
                    "location": [center_x, center_y],
                    "orientation": edge_data['orientation'],
                    "nodes": ordered_nodes,
                    "value": edge_data.get('value', '') # OCR Value
                }
            }
            netlist_dict[node_id].append(component_entry)
    return netlist_dict

def generate_cad(netlist_data, base_filename):
    from svg_schematic import Schematic, Resistor, Capacitor, Inductor, Source, Diode, Wire
    import cairosvg

    svg_path = f'output/cad_drawings/{base_filename}.svg'
    png_path = f'output/cad_drawings/{base_filename}.png'

    text_annotations = []

    with Schematic(filename=svg_path):
        unique_components = {}
        raw_x, raw_y, raw_node_centers = [], [], {}

        for node_str, components in netlist_data.items():
            if not components: continue
            cx = sum(c["component"]["location"][0] for c in components) / len(components)
            cy = sum(c["component"]["location"][1] for c in components) / len(components)
            raw_node_centers[node_str] = (cx, cy)

            for item in components:
                comp_data = item["component"]
                loc = tuple(comp_data["location"])
                comp_key = f"{comp_data['type']}_{loc[0]}_{loc[1]}"
                if comp_key not in unique_components:
                    unique_components[comp_key] = comp_data
                    raw_x.append(loc[0])
                    raw_y.append(loc[1])

        def cluster_coords(coords, threshold=50):
            if not coords: return {}
            sorted_coords = sorted(list(set(coords)))
            clusters, current_cluster = {}, [sorted_coords[0]]
            for val in sorted_coords[1:]:
                cluster_avg = sum(current_cluster) / len(current_cluster)
                if abs(val - cluster_avg) <= threshold: current_cluster.append(val)
                else:
                    final_avg = sum(current_cluster) / len(current_cluster)
                    for v in current_cluster: clusters[v] = final_avg
                    current_cluster = [val]
            final_avg = sum(current_cluster) / len(current_cluster)
            for v in current_cluster: clusters[v] = final_avg
            return clusters

        clustered_x = cluster_coords(raw_x)
        clustered_y = cluster_coords(raw_y)

        max_x_in_drawing = max(clustered_x.values()) if clustered_x else 1000

        drawn_symbols = {}
        node_to_pins = {}

        for comp_key, comp_data in unique_components.items():
            symbol_type = comp_data["type"]
            original_loc = comp_data["location"]
            orientation = comp_data["orientation"]
            snap_loc = (clustered_x[original_loc[0]], clustered_y[original_loc[1]])

            if symbol_type == "resistor": sym = Resistor(C=snap_loc, orient=orientation)
            elif symbol_type == "inductor": sym = Inductor(C=snap_loc, orient=orientation)
            elif symbol_type == "capacitor": sym = Capacitor(C=snap_loc, orient=orientation)
            elif symbol_type == "source": sym = Source(C=snap_loc, orient=orientation, kind="vdc")
            elif symbol_type == "diode":
                sym = Diode(C=snap_loc, orient=orientation)
                sym.p, sym.n = sym.a, sym.c
            else: continue

            drawn_symbols[comp_key] = sym

            # --- SVG TEXT ANNOTATION LOGIC ---
            comp_value = comp_data.get("value", "")
            if comp_value:
                if orientation == 'h':
                    text_annotations.append((snap_loc[0] - 15, snap_loc[1] - 30, comp_value))
                else:
                    if snap_loc[0] > max_x_in_drawing - 50:
                        text_annotations.append((snap_loc[0] - 65, snap_loc[1] + 10, comp_value))
                    else:
                        text_annotations.append((snap_loc[0] + 30, snap_loc[1] + 10, comp_value))

            comp_nodes = comp_data["nodes"]
            if len(comp_nodes) >= 2:
                nodeA, nodeB = comp_nodes[0], comp_nodes[1]
                if nodeA not in node_to_pins: node_to_pins[nodeA] = []
                if nodeB not in node_to_pins: node_to_pins[nodeB] = []

                posA = raw_node_centers.get(nodeA, original_loc)
                dist_p_to_A = (sym.p[0]-posA[0])**2 + (sym.p[1]-posA[1])**2
                dist_n_to_A = (sym.n[0]-posA[0])**2 + (sym.n[1]-posA[1])**2

                if dist_p_to_A < dist_n_to_A:
                    node_to_pins[nodeA].append((sym, sym.p, snap_loc))
                    node_to_pins[nodeB].append((sym, sym.n, snap_loc))
                else:
                    node_to_pins[nodeA].append((sym, sym.n, snap_loc))
                    node_to_pins[nodeB].append((sym, sym.p, snap_loc))

        for node_str, pins in node_to_pins.items():
            if len(pins) < 2: continue
            stubbed_pins = []
            vx_list, vy_list, hx_list, hy_list = [], [], [], []

            for sym_obj, pin_coord, comp_center in pins:
                dx, dy = pin_coord[0] - comp_center[0], pin_coord[1] - comp_center[1]
                if dx == 0 and dy == 0: dx = 1
                step_size = 20
                if abs(dx) >= abs(dy):
                    stub_x, stub_y = pin_coord[0] + (step_size if dx >= 0 else -step_size), pin_coord[1]
                    hx_list.append(stub_x); hy_list.append(stub_y)
                    stubbed_pins.append((sym_obj, pin_coord, (stub_x, stub_y), 'H', dx))
                else:
                    stub_x, stub_y = pin_coord[0], pin_coord[1] + (step_size if dy >= 0 else -step_size)
                    vx_list.append(stub_x); vy_list.append(stub_y)
                    stubbed_pins.append((sym_obj, pin_coord, (stub_x, stub_y), 'V', dy))

            Jx = sum(vx_list)/len(vx_list) if vx_list else sum(hx_list)/len(hx_list)
            Jy = sum(hy_list)/len(hy_list) if hy_list else sum(vy_list)/len(vy_list)
            Junction = (Jx, Jy)

            for sym_obj, pin_coord, stub_coord, direction, delta in stubbed_pins:
                path = [pin_coord, stub_coord]
                if direction == 'H':
                    if (delta > 0 and Junction[0] < stub_coord[0]) or (delta < 0 and Junction[0] > stub_coord[0]):
                        detour_y = stub_coord[1] + (40 if Junction[1] > stub_coord[1] else -40)
                        path.extend([(stub_coord[0], detour_y), (Junction[0], detour_y), Junction])
                    else: path.extend([(Junction[0], stub_coord[1]), Junction])
                else:
                    if (delta > 0 and Junction[1] < stub_coord[1]) or (delta < 0 and Junction[1] > stub_coord[1]):
                        detour_x = stub_coord[0] + (40 if Junction[0] > stub_coord[0] else -40)
                        path.extend([(detour_x, stub_coord[1]), (detour_x, Junction[1]), Junction])
                    else: path.extend([(stub_coord[0], Junction[1]), Junction])

                clean_wire = []
                for pt in path:
                    if not clean_wire or clean_wire[-1] != pt: clean_wire.append(pt)
                if len(clean_wire) > 1: Wire(clean_wire)

    # INJECT TEXT NODES INTO SVG BEFORE CONVERSION
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()

    text_tags = ""
    for x, y, val in text_annotations:
        text_tags += f'<text x="{x}" y="{y}" font-family="Arial" font-size="18" fill="darkred" font-weight="bold">{val}</text>\n'

    svg_content = svg_content.replace("</svg>", text_tags + "</svg>")

    with open(svg_path, "w", encoding="utf-8") as f:
        f.write(svg_content)

    cairosvg.svg2png(url=svg_path, write_to=png_path)
    print(f"✅ Clean Schematic (with extracted values) generated: {png_path}")
    return png_path

# 4. PIPELINE
def process_single_circuit(image_path, filename):
    img_raw = cv2.imread(image_path)
    if img_raw is None:
        print(f"   [ERROR] Could not read '{image_path}'!")
        return None

    orig_h, orig_w = img_raw.shape[:2]
    img_resized = cv2.resize(img_raw, (1000, 750), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

    img_blur = cv2.GaussianBlur(img, (5,5), 0)
    imgTres = cv2.adaptiveThreshold(img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 49, 29)
    imgSkel = morphology.skeletonize(cv2.bitwise_not(imgTres) / 255)
    imgSkel = (imgSkel.astype(np.float32) * 255).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (30, 30))
    img_close = cv2.morphologyEx(imgSkel, cv2.MORPH_CLOSE, kernel, iterations=2)
    img_blob = cv2.erode(img_close, np.ones((3, 3), np.uint8), iterations=2)

    contours, _ = cv2.findContours(img_blob, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    detected_comps = []
    comp_idx = 1
    ocr_data_for_img = ocr_master_dict.get(filename, [])

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if max(w, h) < 45: continue

        square = (y, y+h, x-h//2+w//2, x+h//2+w//2) if h > w else (y-w//2+h//2, y+w//2+h//2, x, x+w)
        y1, y2, x1, x2 = square
        crop = img_resized[y1:y2, x1:x2]
        if crop.size == 0: continue

        try:
            pred = model.predict(preprocess_component_crop(crop), verbose=0)
            class_id = int(np.argmax(pred, axis=-1)[0])

            comp_cx, comp_cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0

            detected_comps.append({
                'id': comp_idx,
                'type': CLASS_NAMES[class_id] if class_id < len(CLASS_NAMES) else "Unknown",
                'location': square,
                'orientation': 'v' if h > w else 'h',
                'center': (comp_cx, comp_cy)
            })
            comp_idx += 1
        except Exception:
            pass

    # GLOBAL DISTANCE SORTING
    possible_matches = []
    for comp in detected_comps:
        for t_idx, ocr_item in enumerate(ocr_data_for_img):
            txt_cx, txt_cy = ocr_item["center"]
            dist = math.hypot(comp['center'][0] - txt_cx, comp['center'][1] - txt_cy)
            if dist < 90:
                possible_matches.append((dist, comp['id'], t_idx, ocr_item["text"]))

    possible_matches.sort(key=lambda x: x[0])

    used_texts = set()
    used_comps = set()
    component_results = {}

    for comp in detected_comps:
        component_results[comp['id']] = {
            'type': comp['type'],
            'location': comp['location'],
            'orientation': comp['orientation'],
            'value': ""
        }

    for dist, cid, tid, txt in possible_matches:
        if cid not in used_comps and tid not in used_texts:
            component_results[cid]['value'] = txt
            used_comps.add(cid)
            used_texts.add(tid)

    # --- NODE DETECTION ---
    h_lines, v_lines, intersection_raw = [], [], []
    LINE_THICKNESS = 25

    for cid, cdata in component_results.items():
        y1, y2, x1, x2 = map(int, cdata['location'])
        if cdata['orientation'] == 'h':
            sy, sx1, sx2 = snap_to_grid(y1 + (y2 - y1) // 2, x1)[0], snap_to_grid(y1 + (y2 - y1) // 2, x1)[1], snap_to_grid(y1 + (y2 - y1) // 2, x2)[1]
            h_lines.extend([(0, sy, sx1, sy), (sx2, sy, 1000, sy)])
        else:
            sy1, sy2, sx = snap_to_grid(y1, x1 + (x2 - x1) // 2)[0], snap_to_grid(y2, x1 + (x2 - x1) // 2)[0], snap_to_grid(y1, x1 + (x2 - x1) // 2)[1]
            v_lines.extend([(sx, 0, sx, sy1), (sx, sy2, sx, 750)])

    for hx1, hy, hx2, _ in h_lines:
        for vx, vy1, _, vy2 in v_lines:
            if (min(hx1, hx2) <= vx <= max(hx1, hx2)) and (min(vy1, vy2) <= hy <= max(vy1, vy2)):
                intersection_raw.append(snap_to_grid(hy, vx))

    for i, (h1x1, h1y, h1x2, _) in enumerate(h_lines):
        for j, (h2x1, h2y, h2x2, _) in enumerate(h_lines):
            if i >= j: continue
            if abs(h1y - h2y) <= LINE_THICKNESS:
                if (h1x1 == 0 or h1x2 == 0) != (h2x1 == 0 or h2x2 == 0):
                    min_x, max_x = max(min(h1x1, h1x2), min(h2x1, h2x2)), min(max(h1x1, h1x2), max(h2x1, h2x2))
                    if min_x <= max_x: intersection_raw.append(snap_to_grid((h1y + h2y) // 2, (min_x + max_x) // 2))

    for i, (v1x, v1y1, _, v1y2) in enumerate(v_lines):
        for j, (v2x, v2y1, _, v2y2) in enumerate(v_lines):
            if i >= j: continue
            if abs(v1x - v2x) <= LINE_THICKNESS:
                if (v1y1 == 0 or v1y2 == 0) != (v2y1 == 0 or v2y2 == 0):
                    min_y, max_y = max(min(v1y1, v1y2), min(v2y1, v2y2)), min(max(v1y1, v1y2), max(v2y1, v2y2))
                    if min_y <= max_y: intersection_raw.append(snap_to_grid((min_y + max_y) // 2, (v1x + v2x) // 2))

    filtered_nodes = []
    nid = 1
    for y1, x1 in sorted(list(set(intersection_raw))):
        if not any(math.hypot(n['location'][1] - x1, n['location'][0] - y1) < 150 for n in filtered_nodes):
            filtered_nodes.append({'name': f"N{nid}", 'location': (y1, x1)})
            nid += 1

    G_ext = nx.Graph()
    for n in filtered_nodes: G_ext.add_node(n['name'])
    for cid, cdata in component_results.items():
        conn = []
        for ty, tx in get_terminal_locations(cdata):
            if filtered_nodes:
                conn.append(min(filtered_nodes, key=lambda n: euclidean((ty, tx), n['location']))['name'])
        if len(conn) == 2:
            G_ext.add_edge(conn[0], conn[1], comp_id=cid, type=cdata['type'], mapped_type=COMP_MAP.get(cdata['type'], "unknown"),
                           location=cdata['location'], orientation=cdata['orientation'], value=cdata['value'])

    img_debug = img_resized.copy()
    for cid, cdata in component_results.items():
        y1, y2, x1, x2 = map(int, cdata['location'])
        cv2.rectangle(img_debug, (x1, y1), (x2, y2), (255, 0, 0), 2)
        if cdata['value']:
            cv2.putText(img_debug, cdata['value'], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    for hx1, hy, hx2, _ in h_lines: cv2.line(img_debug, (hx1, hy), (hx2, hy), (0, 255, 0), 2)
    for vx, vy1, _, vy2 in v_lines: cv2.line(img_debug, (vx, vy1), (vx, vy2), (0, 165, 255), 2)
    for n in filtered_nodes:
        node_y, node_x = n['location']
        cv2.circle(img_debug, (node_x, node_y), 8, (0, 0, 255), -1)

    return {
        "G_pred": G_ext, "pred_nodes": filtered_nodes,
        "components": component_results, "debug_img": img_debug,
        "orig_size": (orig_w, orig_h),
        "netlist_dict": generate_json_netlist(G_ext, filtered_nodes)
    }

# 5. EXECUTION LOOP
image_files = [f for f in os.listdir(TEST_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

node_metrics, total_circuits = [], 0
sum_node_acc, sum_comp_acc = 0, 0
struct_matches, exact_matches = 0, 0

print("\n--- 🚀 PROCESSING SCHEMATICS AND EXTRACTING NETLISTS ---\n")

for filename in image_files:
    img_path = os.path.join(TEST_FOLDER, filename)
    print(f"\n{'='*50}\n⚙️ PROCESSING: {filename}\n{'='*50}")

    res = process_single_circuit(img_path, filename)
    if res is None: continue

    total_circuits += 1
    base_name = filename.split('.')[0]

    json_filename = os.path.join("output/netlists", f"netlist_{base_name}.json")
    with open(json_filename, "w") as f:
        json.dump(res["netlist_dict"], f, indent=4)
    print(f" 📂 Extracted Netlist Saved: {json_filename}")

    try:
        cad_png_path = generate_cad(res["netlist_dict"], base_filename=f"cad_{base_name}")

        cad_img = cv2.imread(cad_png_path)
        cad_img_rgb = cv2.cvtColor(cad_img, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(cv2.cvtColor(res["debug_img"], cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"System Detection with Values - {filename}", fontsize=12, fontweight='bold')
        axes[0].axis("off")

        axes[1].imshow(cad_img_rgb)
        axes[1].set_title("Reconstructed Schematic with Extracted Values", fontsize=12, fontweight='bold')
        axes[1].axis("off")
        plt.show()

    except Exception as e:
        print(f" ❌ Error drawing CAD: {e}")

    # --- METRIC CALCULATIONS ---
    gt_data = ground_truth_db.get(filename, None)
    if gt_data:
        orig_w, orig_h = res["orig_size"]
        gt_count, pred_count, correct, n_acc = calc_node_accuracy(res["pred_nodes"], gt_data["nodes_coords"], orig_w, orig_h, threshold=40)
        comp_acc = calc_component_accuracy(res["components"], gt_data["graph"])
        is_struct, is_exact = eval_isomorphism(res["G_pred"], gt_data["graph"])

        node_metrics.append({
            "Circuit": filename, "GT Nodes": gt_count, "Pred Nodes": pred_count,
            "Correct": correct, "Accuracy (%)": f"{n_acc:.1f}%"
        })

        if is_struct: struct_matches += 1
        if is_exact: exact_matches += 1
        sum_node_acc += n_acc
        sum_comp_acc += comp_acc

        # A. COMPONENT COMPARISON
        gt_types = [data.get('type') for u, v, data in gt_data["graph"].edges(data=True)]
        pred_types = [cdata.get('type') for cdata in res["components"].values()]

        gt_counter, pred_counter = Counter(gt_types), Counter(pred_types)
        all_types = set(gt_counter.keys()).union(set(pred_counter.keys()))

        comp_diag = []
        for ctype in all_types:
            g_cnt = gt_counter.get(ctype, 0)
            p_cnt = pred_counter.get(ctype, 0)
            status = "✅ Match" if g_cnt == p_cnt else ("❌ Mismatch (Count)" if p_cnt > 0 else "❌ Not Found")
            comp_diag.append({"Component Type": ctype, "Ground Truth": g_cnt, "Prediction": p_cnt, "Status": status})

        print(f"\n▶ Component Analysis:")
        display(pd.DataFrame(comp_diag))

        # B. NODE COMPARISON
        gt_scaled = scale_gt_coords(gt_data["nodes_coords"], orig_w, orig_h)
        node_diag = []
        matched_preds = set()

        for i, (gy, gx) in enumerate(gt_scaled):
            min_d, best_pred, best_idx = float('inf'), None, -1
            for j, p in enumerate(res["pred_nodes"]):
                if j in matched_preds: continue
                py, px = p['location']
                d = euclidean((py, px), (gy, gx))
                if d < min_d:
                    min_d, best_pred, best_idx = d, p, j

            if min_d <= 40:
                matched_preds.add(best_idx)
                status = f"✅ Found (Error: {min_d:.1f}px)"
                pred_loc = best_pred['location']
            else:
                status = "❌ Missed"
                pred_loc = "None"

            node_diag.append({"GT Node (Y,X)": (gy, gx), "Prediction (Y,X)": pred_loc, "Status": status})

        print(f"\n▶ Node Analysis:")
        display(pd.DataFrame(node_diag))

# ==========================================
# OVERALL SUMMARY TABLES & ERROR ANALYSIS
# ==========================================
if total_circuits > 0:
    print("\n" + "="*60)
    print("1. END-TO-END PIPELINE ACCURACY")
    print("="*60)
    avg_node = sum_node_acc / total_circuits
    avg_comp = sum_comp_acc / total_circuits
    struct_rate = (struct_matches / total_circuits) * 100
    exact_rate = (exact_matches / total_circuits) * 100

    summary_data = [
        {"Metric": "Component classification accuracy", "Result": f"{avg_comp:.1f}%"},
        {"Metric": "Node detection accuracy", "Result": f"{avg_node:.1f}%"},
        {"Metric": "Structural graph accuracy", "Result": f"{struct_rate:.1f}% ({struct_matches}/{total_circuits})"},
        {"Metric": "Exact circuit reconstruction", "Result": f"{exact_rate:.1f}% ({exact_matches}/{total_circuits})"}
    ]
    display(pd.DataFrame(summary_data))

    print("\n" + "="*60)
    print("2. GLOBAL COMPONENT ERROR ANALYSIS")
    print("="*60)

    global_gt_counter = Counter()
    global_pred_counter = Counter()
    global_correct_counter = Counter()

    for filename in image_files:
        gt_data = ground_truth_db.get(filename, None)
        if not gt_data: continue

        # Yeniden oku (Global analiz için)
        res = process_single_circuit(os.path.join(TEST_FOLDER, filename), filename)
        if not res: continue

        gt_types = [data.get('type') for u, v, data in gt_data["graph"].edges(data=True)]
        pred_types = [cdata.get('type') for cdata in res["components"].values()]

        img_gt_counter = Counter(gt_types)
        img_pred_counter = Counter(pred_types)

        global_gt_counter.update(img_gt_counter)
        global_pred_counter.update(img_pred_counter)

        for ctype in img_gt_counter:
            global_correct_counter[ctype] += min(img_gt_counter[ctype], img_pred_counter[ctype])

    comp_error_data = []
    all_global_types = set(global_gt_counter.keys()).union(set(global_pred_counter.keys()))

    for ctype in all_global_types:
        gt_tot = global_gt_counter.get(ctype, 0)
        pred_tot = global_pred_counter.get(ctype, 0)
        corr = global_correct_counter.get(ctype, 0)
        missed = gt_tot - corr
        false_alarm = pred_tot - corr
        acc = (corr / gt_tot * 100) if gt_tot > 0 else 0.0

        comp_error_data.append({
            "Component Class": ctype,
            "Total GT": gt_tot,
            "Correctly Found": corr,
            "Missed (False Neg)": missed,
            "False Alarms (False Pos)": false_alarm,
            "Accuracy": f"{acc:.1f}%"
        })

    df_comp_err = pd.DataFrame(comp_error_data).sort_values(by="Missed (False Neg)", ascending=False)
    display(df_comp_err)